In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix



pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', None)

# Ignorer tous les avertissements (utilisez avec précaution)
warnings.filterwarnings("ignore")

In [2]:
def contaminant(x):
    if x in ["Neige fraiche (B)", "Neige fraiche (G)"]:
        return "Neige fraiche"
    elif x in ["N.C. (G)","N.C. (B)"]:
        return "N.C."
    else:
        return x
        
df = pd.read_csv('Data_Modelisation-3.csv')
df.Vitesse = df.Vitesse.astype(str)

In [3]:
df = df[df.Contaminant != "DRY"]
df["contaminant"] = df.Contaminant.apply(contaminant)

df = df.drop(['Moyenne CFL F', 'Fv (kg)', 'Tsurf (C)', 'T° BDR (C)', 'Contaminant','hauteur (mm)'], axis=1)

In [4]:
def scale_encode(x, y=0):
    encoder = OneHotEncoder(drop='first')
    # Séparer les variables qualitatives et numériques
    qualitative_vars = ['Vitesse']
    numeric_vars = x.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    # Appliquer le one-hot encoding aux variables qualitatives
    encoded_vars = encoder.fit_transform(x[qualitative_vars]).toarray()
    encoded_columns = encoder.get_feature_names_out(qualitative_vars)
    encoded_df = pd.DataFrame(encoded_vars, columns=encoded_columns, index=x.index)
    
    if y == 1:  # Appliquer le StandardScaler aux variables numériques
        scaler = StandardScaler().set_output(transform='pandas')
    elif y == 2:  # Appliquer le Normalizer aux variables numériques
        scaler = Normalizer().set_output(transform='pandas')
    else:
        X = pd.concat([x.drop(qualitative_vars, axis=1), encoded_df], axis=1)
        return X
    
    # Appliquer le scaler et transformer les variables numériques
    scaled_df = scaler.fit_transform(x[numeric_vars])
    
    # Concaténation des DataFrames encodé et standardisé
    X = pd.concat([scaled_df, encoded_df], axis=1)
    return X

In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Prétraitement des données
df_filtered = df[df["contaminant"] != "Verglas"]
X = df_filtered[['Vitesse', 'Moyenne CFL C', 'Moyenne G% [%]']].copy()
y = df_filtered['contaminant']

# Standardisation des données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ajout des nouvelles caractéristiques
X_scaled = pd.DataFrame(X_scaled, columns=['Vitesse_65', 'Moyenne CFL C', 'Moyenne G% [%]'])
X_scaled["V*CFL"] = X_scaled['Vitesse_65'] * X_scaled["Moyenne CFL C"]
X_scaled["g*CFL"] = X_scaled['Moyenne G% [%]'] * X_scaled["Moyenne CFL C"]
X_scaled["g*V*CFL"] = X_scaled['Moyenne G% [%]'] * X_scaled['Vitesse_65'] * X_scaled["Moyenne CFL C"]

#
# Application de KMeans
kmeans = KMeans(n_clusters=8, random_state=123)
kmeans.fit(X)
y_pred = kmeans.predict(X)

# Mapper les clusters aux classes réelles
def map_clusters_to_labels(y_true, y_clusters):
    mapping = {}
    for cluster in set(y_clusters):
        mask = (y_clusters == cluster)
        most_common = y_true[mask].mode()[0]
        mapping[cluster] = most_common
    return pd.Series(y_clusters).map(mapping)

# Appliquer le mapping
y_train_pred_mapped = map_clusters_to_labels(y, y_pred)
y_test_pred_mapped = map_clusters_to_labels(y_test, y_test_pred)

# Tableau croisé entre les classes réelles et les classes prédites pour l'ensemble de test
confusion = pd.crosstab(y_test, y_test_pred_mapped, rownames=['Actual'], colnames=['Predicted'])

print(confusion)
print(classification_report(y_test, y_test_pred_mapped))


Predicted      Glace  NC.C.  Neige fraiche
Actual                                    
Neige fraiche      8     19             17
               precision    recall  f1-score   support

        Glace       0.27      0.33      0.30        24
         N.C.       0.00      0.00      0.00        54
        NC.C.       0.37      0.61      0.46        51
Neige fraiche       0.50      0.70      0.58        71
        SLUSH       0.00      0.00      0.00        15

     accuracy                           0.41       215
    macro avg       0.23      0.33      0.27       215
 weighted avg       0.28      0.41      0.33       215

